<a href="https://colab.research.google.com/github/ahmykhan/WESAD-Stress-Classification/blob/main/AI_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

# This lists all CSV files inside your Google Drive WESAD folder
print("Searching for your CSV files...")
for root, dirs, files in os.walk('/content/drive/Othercomputers/My Laptop/archive/WESAD'):
    for file in files:
        if file.endswith('.csv'):
            print(os.path.join(root, file))

Searching for your CSV files...
/content/drive/Othercomputers/My Laptop/archive/WESAD/S10/S10_quest.csv
/content/drive/Othercomputers/My Laptop/archive/WESAD/S10/S10_E4_Data/BVP.csv
/content/drive/Othercomputers/My Laptop/archive/WESAD/S10/S10_E4_Data/ACC.csv
/content/drive/Othercomputers/My Laptop/archive/WESAD/S10/S10_E4_Data/EDA.csv
/content/drive/Othercomputers/My Laptop/archive/WESAD/S10/S10_E4_Data/HR.csv
/content/drive/Othercomputers/My Laptop/archive/WESAD/S10/S10_E4_Data/TEMP.csv
/content/drive/Othercomputers/My Laptop/archive/WESAD/S10/S10_E4_Data/tags.csv
/content/drive/Othercomputers/My Laptop/archive/WESAD/S10/S10_E4_Data/IBI.csv
/content/drive/Othercomputers/My Laptop/archive/WESAD/S11/S11_quest.csv
/content/drive/Othercomputers/My Laptop/archive/WESAD/S11/S11_E4_Data/ACC.csv
/content/drive/Othercomputers/My Laptop/archive/WESAD/S11/S11_E4_Data/EDA.csv
/content/drive/Othercomputers/My Laptop/archive/WESAD/S11/S11_E4_Data/BVP.csv
/content/drive/Othercomputers/My Laptop/arc

In [ ]:
import pandas as pd

# Paste your full CSV path below:
file_path = '/content/drive/Othercomputers/My Laptop/archive/WESAD/S2/S2_E4_Data/BVP.csv'  # <--- Update path here

# Load the first 5 rows
df = pd.read_csv(file_path, nrows=5)
print("Column names in your CSV:")
print(df.columns.tolist())
df.head()

Column names in your CSV:
['1495437325.00']


,1495437325.00
0,64.0
1,-0.0
2,-0.0
3,-0.0
4,-0.0


In [ ]:
import pickle
import pandas as pd

# Paste your copied path between the quotes below:
file_path = '/content/drive/Othercomputers/My Laptop/archive/WESAD/S2/S2.pkl'  # e.g., '/content/drive/MyDrive/.../S2/S2.pkl'

print("Loading S2.pkl...")
with open(file_path, 'rb') as f:
    data = pickle.load(f, encoding='latin1')

# Extract ECG, EDA, and status labels from chest sensor array
ecg = data['signal']['chest']['ECG'].flatten()
eda = data['signal']['chest']['EDA'].flatten()
labels = data['label'].flatten()

# Combine into a dataframe
df = pd.DataFrame({
    'Heartbeat': ecg,
    'Sweat': eda,
    'Status': labels
})

# Keep only Relaxed (1) and Stressed (2) states
df_clean = df[df['Status'].isin([1, 2])].copy()
df_clean['Status'] = df_clean['Status'].map({1: 'Relaxed', 2: 'Stressed'})

# Save lightweight CSV directly to Colab environment
output_name = 'Cleaned_Person2_Data.csv'
df_clean.to_csv(output_name, index=False)

print(f"✅ SUCCESS! Created '{output_name}'")
print("\nFirst 5 rows of your cleaned dataset:")
print(df_clean.head())

Loading S2.pkl...
✅ SUCCESS! Created 'Cleaned_Person2_Data.csv'

First 5 rows of your cleaned dataset:
        Heartbeat     Sweat   Status
214583   0.030945  5.710983  Relaxed
214584   0.033646  5.719376  Relaxed
214585   0.033005  5.706406  Relaxed
214586   0.031815  5.712509  Relaxed
214587   0.030350  5.727005  Relaxed


In [1]:
import os
import pickle
import numpy as np
import pandas as pd

# 1. Setup paths and subjects (Excluding S1 and S12 per dataset documentation)
drive_path = '/content/drive/Othercomputers/My Laptop/archive/WESAD'  # Update this if your folder name is different
subjects = [f'S{i}' for i in range(2, 18) if i != 12]

# 2. Define windowing parameters (700 Hz sampling rate * 5 seconds = 3500 rows per window)
fs = 700
window_samples = fs * 5
master_data = []

print("🚀 Starting Phase 2 End-to-End Preprocessing...")

for sub in subjects:
    file_path = os.path.join(drive_path, sub, f'{sub}.pkl')

    if not os.path.exists(file_path):
        print(f"⚠️ Missing {sub}.pkl. Skipping...")
        continue

    print(f"Processing {sub}...")
    with open(file_path, 'rb') as f:
        data = pickle.load(f, encoding='latin1')

    # Extract raw chest signals and labels
    ecg = data['signal']['chest']['ECG'].flatten()
    eda = data['signal']['chest']['EDA'].flatten()
    labels = data['label'].flatten()

    # Create a temporary dataframe for the subject
    df = pd.DataFrame({'ECG': ecg, 'EDA': eda, 'Label': labels})

    # Filter only Baseline/Relaxed (1) and Stressed (2)
    df = df[df['Label'].isin([1, 2])].reset_index(drop=True)

    # 3. Feature Extraction (Compressing 700Hz data into 5-second windows)
    total_rows = len(df)
    for start_idx in range(0, total_rows - window_samples, window_samples):
        end_idx = start_idx + window_samples
        window = df.iloc[start_idx:end_idx]

        # Ensure the window has a consistent label (take the most frequent label in this 5s block)
        majority_label = window['Label'].mode()[0]
        state = 'Relaxed' if majority_label == 1 else 'Stressed'

        # Calculate statistical features for this 5-second block
        master_data.append({
            'Subject': sub,
            'State': state,
            'ECG_Mean': window['ECG'].mean(),
            'ECG_Std': window['ECG'].std(),
            'ECG_Max': window['ECG'].max(),
            'ECG_Min': window['ECG'].min(),
            'EDA_Mean': window['EDA'].mean(),
            'EDA_Std': window['EDA'].std(),
            'EDA_Max': window['EDA'].max(),
            'EDA_Min': window['EDA'].min()
        })

# 4. Finalize and Export
final_df = pd.DataFrame(master_data)
output_file = '/content/drive/MyDrive/WESAD_ML_Ready_Features.csv'
final_df.to_csv(output_file, index=False)

print(f"\n✅ Phase 2 Complete! Master dataset saved to: {output_file}")
print(f"Total machine-learning-ready rows: {len(final_df)}")
print("\nDataset Preview:")
print(final_df.head())

🚀 Starting Phase 2 End-to-End Preprocessing...
Processing S2...
Processing S3...
Processing S4...
Processing S5...
Processing S6...
Processing S7...
Processing S8...
Processing S9...
Processing S10...
Processing S11...
Processing S13...
Processing S14...
Processing S15...
Processing S16...
Processing S17...

✅ Phase 2 Complete! Master dataset saved to: /content/drive/MyDrive/WESAD_ML_Ready_Features.csv
Total machine-learning-ready rows: 5504

Dataset Preview:
  Subject    State  ECG_Mean   ECG_Std   ECG_Max   ECG_Min  EDA_Mean  \
0      S2  Relaxed  0.008192  0.129848  0.979294 -0.220459  5.472599   
1      S2  Relaxed -0.000039  0.139631  0.957870 -0.222336  5.237507   
2      S2  Relaxed  0.003330  0.125532  0.817657 -0.247375  5.166873   
3      S2  Relaxed  0.001545  0.123407  0.778381 -0.293884  5.127708   
4      S2  Relaxed -0.001368  0.124402  0.767166 -0.235245  5.036110   

    EDA_Std   EDA_Max   EDA_Min  
0  0.107337  5.756760  5.310822  
1  0.056015  5.378342  5.145645  
2